# [INFO-H515 - Big Data Scalable Analytics](https://uv.ulb.ac.be/course/view.php?id=85246?username=guest)

## TP 5 - Examn question : Inner Product

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*

####  2025


## General imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession

## Data generation

In [2]:
def genData(N, n, random_seed):
    np.random.seed(0)

    #Inputs and the weights of the linear combination are drawn at random
    X = np.random.rand(N, n)

    print("Number of observations :", N)
    print("Number of features :", n)

    print("Dimension of X :", X.shape)

    return X

In [128]:
X = genData(500, 10, 42)

Number of observations : 500
Number of features : 10
Dimension of X : (500, 10)


In [129]:
data = X
data = np.concatenate((np.arange(len(data)).reshape(1, -1).transpose(), data), axis=1)

In [130]:
data.shape

(500, 11)

In [131]:
pd.DataFrame(data).head()

,0,1,2,3,4,5,6,7,8,9,10
0,0.0,0.548814,0.715189,0.602763,0.544883,0.423655,0.645894,0.437587,0.891773,0.963663,0.383442
1,1.0,0.791725,0.528895,0.568045,0.925597,0.071036,0.087129,0.020218,0.832620,0.778157,0.870012
2,2.0,0.978618,0.799159,0.461479,0.780529,0.118274,0.639921,0.143353,0.944669,0.521848,0.414662
3,3.0,0.264556,0.774234,0.456150,0.568434,0.018790,0.617635,0.612096,0.616934,0.943748,0.681820
4,4.0,0.359508,0.437032,0.697631,0.060225,0.666767,0.670638,0.210383,0.128926,0.315428,0.363711


## Spark Session

In [132]:
# Initialize Spark Session (local mode)
# Start Spark session with local master and 2 cores
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("LinearRegression") \
    .getOrCreate()

# Let us retrieve the sparkContext object
sc = spark.sparkContext

spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [133]:
#Let us use 2 partitions
B = 2
X_RDD = sc.parallelize(data, B).cache()

In [134]:
np.array(X_RDD.collect()).shape

(500, 11)

**Exercise:**

Given two vectors of size $n$, $x_1 = [x_{11}, \cdots, x_{1n}]$ and $x_2 = [x_{21}, \cdots, x_{2n}]$, their inner product is
$\langle x_1, x_2 \rangle = \sum_{i=1}^n x_{1i} x_{2i}$

Let us consider a dataset of $N$ vectors (with huge $N$) of size $n << N$ stored in a row-wise manner as follows

\begin{equation}
\begin{pmatrix}
  0       & X_{1,1}   & X_{1,2}   & \cdots  & X_{1,n}  \\
  1       & X_{2,1}   & X_{2,2}   & \cdots  & X_{2,n} \\
  \cdots \\
  N       & X_{N,1}   & X_{N,2}   & \cdots  & X_{N,n} \\
\end{pmatrix}
\end{equation}


Suppose we want to compute by map-reduce all possible inner products
$\langle x_k, x_j \rangle$ $k = 1, \cdots,  N$, $j = 1, \cdots,  N$
where $x_k$ denotes the vector in the k-th row.


Write a map-reduce code (RDD-based) to compute all possible pairwise inner products.
Note that given the huge value $N$, no transpose of the dataset is possible

**Your Solution**

**Solution**

In [135]:
N = 500
n = 10

In [136]:
# 1. create one line per pair of sample and feature,containing the ids of the samples and features as key, and the value of the feature, for one of the sample
a = X_RDD.flatMap(lambda x: [[((int(x[0]), i, j), x[j+1]) for i in range(N)] + [((i, int(x[0]), j), x[j+1]) for i in range(N)] for j in range(n)]).flatMap(lambda x: x)

# 2. group the lines by key (pair of samples and correspondig features) and compute the product
b = a.reduceByKey(lambda x, y: x*y)

# 3. remove the feature id
c = a.map(lambda x: ((x[0][0], x[0][1]), x[1]))

# 4. compute the sum of products for each pair of samples
d = c.reduceByKey(lambda x, y: x+y)


# --- to create a matrix of size N,N

# 5. keep the id of the first sample of the pair as key, and put the id of the other sample in the value. Create of list of one tuple with the value
e = d.map(lambda x : (x[0][0], [(x[0][1], x[1])]))

# 6. concatenate the tuples from all pairs where the first sample has the same id
f = e.reduceByKey(lambda x, y: x+y)

# 7. for each line (inner products of 2 vectors where the first has the id of the line), sort the values according to the id of the second sample and remove this ID
# Then, sort the lines according to the id of the first sample and remove this id
g = f.map(lambda x : (x[0], [elem[1] for elem in sorted(x[1], key = lambda y:y[0])])).sortByKey().map(lambda x:x[1])

In [139]:
pd.DataFrame(d.collect())

,0,1
0,"(0, 0)",12.315326
1,"(0, 2)",11.960176
2,"(0, 4)",10.067912
3,"(0, 6)",9.638965
4,"(0, 8)",10.600951
...,...,...
249995,"(499, 496)",6.570955
249996,"(497, 498)",11.745189
249997,"(498, 497)",11.745189
249998,"(498, 499)",8.923166


In [137]:
pd.DataFrame(g.collect())

,0,1,2,3,4,5,6,7,8,9,...,490,491,492,493,494,495,496,497,498,499
0,12.315326,11.631096,11.960176,11.712060,10.067912,10.244202,9.638965,10.780703,10.600951,9.904621,...,11.132584,11.235725,11.827334,9.547250,12.118846,9.982870,9.223576,12.484727,11.575787,9.662705
1,11.631096,10.946867,11.275947,11.027831,9.383683,9.559973,8.954736,10.096474,9.916722,9.220391,...,10.448355,10.551496,11.143105,8.863021,11.434617,9.298641,8.539347,11.800498,10.891558,8.978476
2,11.960176,11.275947,11.605027,11.356910,9.712762,9.889053,9.283816,10.425553,10.245801,9.549471,...,10.777435,10.880575,11.472185,9.192100,11.763697,9.627720,8.868426,12.129577,11.220638,9.307555
3,11.712060,11.027831,11.356910,11.108794,9.464646,9.640936,9.035700,10.177437,9.997685,9.301355,...,10.529318,10.632459,11.224069,8.943984,11.515580,9.379604,8.620310,11.881461,10.972521,9.059439
4,10.067912,9.383683,9.712762,9.464646,7.820498,7.996789,7.391552,8.533289,8.353537,7.657207,...,8.885170,8.988311,9.579921,7.299836,9.871432,7.735456,6.976162,10.237313,9.328374,7.415291
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,9.982870,9.298641,9.627720,9.379604,7.735456,7.911746,7.306510,8.448247,8.268495,7.572165,...,8.800128,8.903269,9.494878,7.214794,9.786390,7.650414,6.891120,10.152271,9.243331,7.330249
496,9.223576,8.539347,8.868426,8.620310,6.976162,7.152453,6.547216,7.688953,7.509201,6.812871,...,8.040834,8.143975,8.735585,6.455500,9.027096,6.891120,6.131826,9.392977,8.484038,6.570955
497,12.484727,11.800498,12.129577,11.881461,10.237313,10.413604,9.808367,10.950104,10.770352,10.074022,...,11.301986,11.405126,11.996736,9.716651,12.288247,10.152271,9.392977,12.654128,11.745189,9.832106
498,11.575787,10.891558,11.220638,10.972521,9.328374,9.504664,8.899427,10.041165,9.861412,9.165082,...,10.393046,10.496187,11.087796,8.807711,11.379308,9.243331,8.484038,11.745189,10.836249,8.923166


This solution creates 2N²n tuples at step 1